In [ ]:
# import os
# import base64
# import glob
# import csv
# from openai import OpenAI
# from concurrent.futures import ThreadPoolExecutor, as_completed

# # ==========================================
# # 1. 配置参数与路径设置
# # ==========================================

# ApiKey = "00ZI8D8IDY1A826721992EAU8S5KG6QJQ5575ZFE3YHGYV17"

# client = OpenAI(
#     api_key=ApiKey,
#     base_url="https://itest.clife.net/agentpaas/paas-llm/v1",
#     default_headers={
#         "Accept": "*/*",
#         "ApiKey": ApiKey
#     },
# )

# MODEL_NAME = "qwen3.6-plus"
# MAX_WORKERS = 5  # 设置并发线程数

# # 目录配置
# INPUT_DIR = "data/pic"                      # 赛方规定的图片输入目录
# CACHE_DIR = "cache_results"            # 缓存目录，用于断点续传
# OUTPUT_DIR = "submission"              # 最终提交包的目录
# OUTPUT_CSV = os.path.join(OUTPUT_DIR, "submission.csv")

# # ==========================================
# # 2. 核心提示词定义 (System Prompt)
# # ==========================================
# SYSTEM_PROMPT = """你是一个顶级的计算化学专家和具有极高准确率的光学化学结构识别（OCSR）工程师。
# 你的任务是读取一张分子结构示意图，准确识别其中的分子，并严格按照 Extended-SMILES (E-SMILES) 规范输出结果。

# ### Extended-SMILES (E-SMILES) 规范要求（必须严格遵守）：
# 基本格式必须为：`SMILES<sep>EXTENSION`
# 1. **SMILES**：前半部分是与 RDKit 兼容的标准 SMILES 字符串。
# 2. **<sep>**：作为特殊分隔符，将常规 SMILES 与扩展描述分开（若无扩展结构，视情况保留或省略均可，但建议统一格式）。
# 3. **EXTENSION**：后半部分使用类似 XML 的特殊标记来描述复杂结构（如马库什结构等）：
#    - 取代基/缩写基团: 格式为 `<a> [ATOM_INDEX]: [GROUP_NAME] </a>`。例如：`<a>0:R[1]</a>` 或 `<a>12:Ph</a>`。
#    - 位置不确定的环取代物: 格式为 `<r> [RING_INDEX]: [GROUP_NAME] </r>`。
#    - 抽象环: 格式为 `<c> [CIRCLE_INDEX]: [CIRCLE_NAME] </c>`。
#    - 连接点: 使用特殊标记 `<dum>`，如 `<a>0:<dum></a>`。
   
# *(注：[ATOM_INDEX] 和 [RING_INDEX] 等索引均从 0 开始。)*

# ### 输出格式严格约束：
# - **直接且仅输出**合法的 E-SMILES 字符串文本。
# - **绝对不要**输出任何多余的解释性文字。
# - **绝对不要**使用 Markdown 格式（如 ``` 符号）。

# 输出示例参考：
# *c1ccccc1<sep><a>0:R[1]</a>
# """

# # ==========================================
# # 3. 辅助函数：图片转 Base64
# # ==========================================
# def encode_image_to_base64(image_path):
#     with open(image_path, "rb") as image_file:
#         return base64.b64encode(image_file.read()).decode('utf-8')

# # ==========================================
# # 4. 单张图片处理函数
# # ==========================================
# def process_single_image(image_path, cache_path):
#     base64_image = encode_image_to_base64(image_path)
#     print(f"🚀 线程启动: 正在处理 {os.path.basename(image_path)} ...")
    
#     try:
#         response = client.chat.completions.create(
#             model=MODEL_NAME,
#             messages=[
#                 {
#                     "role": "system",
#                     "content": SYSTEM_PROMPT
#                 },
#                 {
#                     "role": "user",
#                     "content": [
#                         {
#                             "type": "text",
#                             "text": "请识别这张图片中的分子，并直接返回该分子的 Extended-SMILES 字符串。"
#                         },
#                         {
#                             "type": "image_url",
#                             "image_url": {
#                                 "url": f"data:image/jpeg;base64,{base64_image}"
#                             }
#                         }
#                     ]
#                 }
#             ],
#             temperature=0.1, 
#         )
        
#         # 提取 Token 消耗
#         prompt_tokens = getattr(response.usage, 'prompt_tokens', 0) if hasattr(response, 'usage') else 0
#         completion_tokens = getattr(response.usage, 'completion_tokens', 0) if hasattr(response, 'usage') else 0
        
#         # 提取并清理文本
#         result_text = response.choices[0].message.content
#         clean_e_smiles = result_text.replace("```", "").strip()
        
#         # 写入 txt 缓存文件
#         with open(cache_path, 'w', encoding='utf-8') as f:
#             f.write(clean_e_smiles)
            
#         print(f"✅ 成功: {os.path.basename(image_path)} -> 缓存完成 (消耗: {prompt_tokens} in / {completion_tokens} out)")
#         return True, prompt_tokens, completion_tokens
        
#     except Exception as e:
#         print(f"❌ 发生错误 ({os.path.basename(image_path)}): {e}")
        
#     return False, 0, 0

# # ==========================================
# # 5. 汇总生成 CSV 文件
# # ==========================================
# def generate_submission_csv(image_files):
#     print(f"\n" + "="*50)
#     print(f"📄 开始生成最终的 submission.csv ...")
    
#     os.makedirs(OUTPUT_DIR, exist_ok=True)
    
#     with open(OUTPUT_CSV, mode='w', encoding='utf-8', newline='') as f:
#         writer = csv.writer(f)
#         # 写入表头（严格遵循赛方规范）
#         writer.writerow(['file_name', 'e_smiles'])
        
#         for img_path in image_files:
#             file_name = os.path.basename(img_path)
#             base_name = os.path.splitext(file_name)[0]
#             cache_path = os.path.join(CACHE_DIR, f"{base_name}.txt")
            
#             e_smiles = ""
#             if os.path.exists(cache_path):
#                 with open(cache_path, 'r', encoding='utf-8') as cf:
#                     e_smiles = cf.read().strip()
            
#             writer.writerow([file_name, e_smiles])
            
#     print(f"✅ 结果已保存至: {OUTPUT_CSV}")

# # ==========================================
# # 6. 主执行循环 (多线程并发控制)
# # ==========================================
# def main():
#     # 确保所需目录存在
#     os.makedirs(CACHE_DIR, exist_ok=True)
#     os.makedirs(OUTPUT_DIR, exist_ok=True)
    
#     if not os.path.exists(INPUT_DIR):
#         print(f"错误：输入文件夹 '{INPUT_DIR}' 不存在。请确保图片放在对应的目录。")
#         return

#     # 获取所有图片文件
#     image_extensions = ('*.png', '*.jpg', '*.jpeg')
#     image_files = []
#     for ext in image_extensions:
#         image_files.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
#         image_files.extend(glob.glob(os.path.join(INPUT_DIR, ext.upper()))) 
    
#     image_files = sorted(list(set(image_files)))
    
#     if not image_files:
#         print(f"在 '{INPUT_DIR}' 中没有找到图片文件。")
#         return
        
#     print(f"找到 {len(image_files)} 张图片，开始多线程批量处理 (最大线程数: {MAX_WORKERS})...\n" + "-"*50)
    
#     success_count = 0
#     total_prompt_tokens = 0
#     total_completion_tokens = 0
    
#     # 阶段一：任务预过滤（断点续传拦截）
#     tasks_to_run = []
#     for img_path in image_files:
#         base_name = os.path.splitext(os.path.basename(img_path))[0]
#         cache_path = os.path.join(CACHE_DIR, f"{base_name}.txt")
        
#         if os.path.exists(cache_path):
#             print(f"⏩ 跳过: {base_name}.txt 缓存已存在。")
#             success_count += 1
#         else:
#             tasks_to_run.append((img_path, cache_path))
            
#     print(f"\n待处理的新任务数: {len(tasks_to_run)}")
#     print("-" * 50)
    
#     # 阶段二：提交给线程池
#     if tasks_to_run:
#         with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#             future_to_img = {
#                 executor.submit(process_single_image, img_path, cache_path): img_path 
#                 for img_path, cache_path in tasks_to_run
#             }
            
#             for future in as_completed(future_to_img):
#                 img_path = future_to_img[future]
#                 try:
#                     is_success, p_tokens, c_tokens = future.result()
#                     if is_success:
#                         success_count += 1
#                         total_prompt_tokens += p_tokens
#                         total_completion_tokens += c_tokens
#                 except Exception as exc:
#                     print(f"❌ 线程池执行异常 ({os.path.basename(img_path)}): {exc}")

#     # 阶段三：合并结果并生成 CSV
#     generate_submission_csv(image_files)

#     # 阶段四：统计与总结
#     print("-" * 50)
#     print(f"🎉 处理完成！共成功处理并合成了 {success_count} / {len(image_files)} 个结果。")
#     if tasks_to_run:
#         print("\n📊 本次运行 Token 消耗统计:")
#         print(f"   - 总输入 Token (Prompt):     {total_prompt_tokens}")
#         print(f"   - 总输出 Token (Completion): {total_completion_tokens}")
#         print(f"   - 总计消耗 Token:            {total_prompt_tokens + total_completion_tokens}\n")
    
#     print(f"⚠️ 提交流程提醒：")
#     print(f"1. 请检查 {OUTPUT_DIR}/submission.csv 数据是否完整。")
#     print(f"2. 请在此目录手动添加或补充要求的 meta.md 文件。")
#     print(f"3. 将上述两份文件直接打成 submission.zip 压缩包上传！")

# if __name__ == "__main__":
#     main()

# 调用本地的vllm大模型

In [1]:
import os
import base64
import glob
import csv
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================================
# 1. 配置参数与路径设置
# ==========================================

# vLLM 本地部署默认不需要鉴权，通常填 "EMPTY" 即可
ApiKey = "EMPTY"

client = OpenAI(
    api_key=ApiKey,
    base_url="http://10.6.14.15:8004/v1", # 修改为本地 vLLM 服务的地址和指定端口 8004
)

# 修改为 vLLM 启动时实际使用的模型路径/名称
MODEL_NAME = "/mnt/sda/zhuangyungui/pretrained_models/Qwen3.5-27B"

MAX_WORKERS = 10  # 设置并发线程数

# 目录配置
INPUT_DIR = "data/pic"                  # 赛方规定的图片输入目录
CACHE_DIR = "cache_results"            # 缓存目录，用于断点续传
OUTPUT_DIR = "submission"              # 最终提交包的目录
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "submission.csv")

# ==========================================
# 2. 核心提示词定义 (System Prompt)
# ==========================================
SYSTEM_PROMPT = """你是一个顶级的计算化学专家和具有极高准确率的光学化学结构识别（OCSR）工程师。
你的任务是读取一张分子结构示意图，准确识别其中的分子，并严格按照 Extended-SMILES (E-SMILES) 规范输出结果。

### Extended-SMILES (E-SMILES) 规范要求（必须严格遵守）：
基本格式必须为：`SMILES<sep>EXTENSION`
1. **SMILES**：前半部分是与 RDKit 兼容的标准 SMILES 字符串。
2. **<sep>**：作为特殊分隔符，将常规 SMILES 与扩展描述分开（若无扩展结构，视情况保留或省略均可，但建议统一格式）。
3. **EXTENSION**：后半部分使用类似 XML 的特殊标记来描述复杂结构（如马库什结构等）：
   - 取代基/缩写基团: 格式为 `<a> [ATOM_INDEX]: [GROUP_NAME] </a>`。例如：`<a>0:R[1]</a>` 或 `<a>12:Ph</a>`。
   - 位置不确定的环取代物: 格式为 `<r> [RING_INDEX]: [GROUP_NAME] </r>`。
   - 抽象环: 格式为 `<c> [CIRCLE_INDEX]: [CIRCLE_NAME] </c>`。
   - 连接点: 使用特殊标记 `<dum>`，如 `<a>0:<dum></a>`。
   
*(注：[ATOM_INDEX] 和 [RING_INDEX] 等索引均从 0 开始。)*

### 输出格式严格约束：
- **直接且仅输出**合法的 E-SMILES 字符串文本。
- **绝对不要**输出任何多余的解释性文字。
- **绝对不要**使用 Markdown 格式（如 ``` 符号）。

输出示例参考：
*c1ccccc1<sep><a>0:R[1]</a>
"""

# ==========================================
# 3. 辅助函数：图片转 Base64
# ==========================================
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# ==========================================
# 4. 单张图片处理函数
# ==========================================
def process_single_image(image_path, cache_path):
    base64_image = encode_image_to_base64(image_path)
    print(f"🚀 线程启动: 正在处理 {os.path.basename(image_path)} ...")
    
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            extra_body={
                "chat_template_kwargs": {"enable_thinking": False},
            }, 
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "请识别这张图片中的分子，并直接返回该分子的 Extended-SMILES 字符串。"
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            temperature=0.1, 
        )
        
        # 提取 Token 消耗
        prompt_tokens = getattr(response.usage, 'prompt_tokens', 0) if hasattr(response, 'usage') else 0
        completion_tokens = getattr(response.usage, 'completion_tokens', 0) if hasattr(response, 'usage') else 0
        
        # 提取并清理文本
        result_text = response.choices[0].message.content
        clean_e_smiles = result_text.replace("```", "").strip()
        
        # 写入 txt 缓存文件
        with open(cache_path, 'w', encoding='utf-8') as f:
            f.write(clean_e_smiles)
            
        print(f"✅ 成功: {os.path.basename(image_path)} -> 缓存完成 (消耗: {prompt_tokens} in / {completion_tokens} out)")
        return True, prompt_tokens, completion_tokens
        
    except Exception as e:
        print(f"❌ 发生错误 ({os.path.basename(image_path)}): {e}")
        
    return False, 0, 0

# ==========================================
# 5. 汇总生成 CSV 文件
# ==========================================
def generate_submission_csv(image_files):
    print(f"\n" + "="*50)
    print(f"📄 开始生成最终的 submission.csv ...")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    with open(OUTPUT_CSV, mode='w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        # 写入表头（严格遵循赛方规范）
        writer.writerow(['file_name', 'e_smiles'])
        
        for img_path in image_files:
            file_name = os.path.basename(img_path)
            base_name = os.path.splitext(file_name)[0]
            cache_path = os.path.join(CACHE_DIR, f"{base_name}.txt")
            
            e_smiles = ""
            if os.path.exists(cache_path):
                with open(cache_path, 'r', encoding='utf-8') as cf:
                    e_smiles = cf.read().strip()
            
            writer.writerow([file_name, e_smiles])
            
    print(f"✅ 结果已保存至: {OUTPUT_CSV}")

# ==========================================
# 6. 主执行循环 (多线程并发控制)
# ==========================================
def main():
    # 确保所需目录存在
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if not os.path.exists(INPUT_DIR):
        print(f"错误：输入文件夹 '{INPUT_DIR}' 不存在。请确保图片放在对应的目录。")
        return

    # 获取所有图片文件
    image_extensions = ('*.png', '*.jpg', '*.jpeg')
    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
        image_files.extend(glob.glob(os.path.join(INPUT_DIR, ext.upper()))) 
    
    image_files = sorted(list(set(image_files)))
    
    if not image_files:
        print(f"在 '{INPUT_DIR}' 中没有找到图片文件。")
        return
        
    print(f"找到 {len(image_files)} 张图片，开始多线程批量处理 (最大线程数: {MAX_WORKERS})...\n" + "-"*50)
    
    success_count = 0
    total_prompt_tokens = 0
    total_completion_tokens = 0
    
    # 阶段一：任务预过滤（断点续传拦截）
    tasks_to_run = []
    for img_path in image_files:
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        cache_path = os.path.join(CACHE_DIR, f"{base_name}.txt")
        
        if os.path.exists(cache_path):
            print(f"⏩ 跳过: {base_name}.txt 缓存已存在。")
            success_count += 1
        else:
            tasks_to_run.append((img_path, cache_path))
            
    print(f"\n待处理的新任务数: {len(tasks_to_run)}")
    print("-" * 50)
    
    # 阶段二：提交给线程池
    if tasks_to_run:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_img = {
                executor.submit(process_single_image, img_path, cache_path): img_path 
                for img_path, cache_path in tasks_to_run
            }
            
            for future in as_completed(future_to_img):
                img_path = future_to_img[future]
                try:
                    is_success, p_tokens, c_tokens = future.result()
                    if is_success:
                        success_count += 1
                        total_prompt_tokens += p_tokens
                        total_completion_tokens += c_tokens
                except Exception as exc:
                    print(f"❌ 线程池执行异常 ({os.path.basename(img_path)}): {exc}")

    # 阶段三：合并结果并生成 CSV
    generate_submission_csv(image_files)

    # 阶段四：统计与总结
    print("-" * 50)
    print(f"🎉 处理完成！共成功处理并合成了 {success_count} / {len(image_files)} 个结果。")
    if tasks_to_run:
        print("\n📊 本次运行 Token 消耗统计:")
        print(f"   - 总输入 Token (Prompt):     {total_prompt_tokens}")
        print(f"   - 总输出 Token (Completion): {total_completion_tokens}")
        print(f"   - 总计消耗 Token:            {total_prompt_tokens + total_completion_tokens}\n")
    
    print(f"⚠️ 提交流程提醒：")
    print(f"1. 请检查 {OUTPUT_DIR}/submission.csv 数据是否完整。")
    print(f"2. 请在此目录手动添加或补充要求的 meta.md 文件。")
    print(f"3. 将上述两份文件直接打成 submission.zip 压缩包上传！")

if __name__ == "__main__":
    main()

找到 4000 张图片，开始多线程批量处理 (最大线程数: 10)...
--------------------------------------------------
⏩ 跳过: mol_0002.txt 缓存已存在。
⏩ 跳过: mol_0010.txt 缓存已存在。

待处理的新任务数: 3998
--------------------------------------------------
🚀 线程启动: 正在处理 mol_0003.png ...
🚀 线程启动: 正在处理 mol_0001.png ...
🚀 线程启动: 正在处理 mol_0004.png ...
🚀 线程启动: 正在处理 mol_0006.png ...
🚀 线程启动: 正在处理 mol_0005.png ...
🚀 线程启动: 正在处理 mol_0007.png ...
🚀 线程启动: 正在处理 mol_0008.png ...
🚀 线程启动: 正在处理 mol_0009.png ...
🚀 线程启动: 正在处理 mol_0011.png ...
🚀 线程启动: 正在处理 mol_0012.png ...
✅ 成功: mol_0001.png -> 缓存完成 (消耗: 527 in / 30 out)
🚀 线程启动: 正在处理 mol_0013.png ...
✅ 成功: mol_0009.png -> 缓存完成 (消耗: 521 in / 32 out)
✅ 成功: mol_0007.png -> 缓存完成 (消耗: 535 in / 33 out)
🚀 线程启动: 正在处理 mol_0015.png ...
🚀 线程启动: 正在处理 mol_0014.png ...
✅ 成功: mol_0012.png -> 缓存完成 (消耗: 521 in / 37 out)
🚀 线程启动: 正在处理 mol_0016.png ...
✅ 成功: mol_0011.png -> 缓存完成 (消耗: 535 in / 49 out)
✅ 成功: mol_0008.png -> 缓存完成 (消耗: 525 in / 50 out)
🚀 线程启动: 正在处理 mol_0018.png ...
🚀 线程启动: 正在处理 mol_0017.png ...
✅ 成功: mol_0003.png 